# iMAKS SBERT F1_content 평가

고정 예측 172건 / 원본 정답 86건. **런타임 → 모두 실행**. 패키지·모델은 Colab 서버에서만 설치됩니다. OpenAI API 키와 Drive 마운트는 필요하지 않습니다.

모델: all-MiniLM-L6-v2, 문서별 Hungarian 1:1 최대 유사도 할당 후 임계값 0.6. 입력은 condition + 공백 + action이며 ID·인용문·클래스는 매칭에 사용하지 않습니다. 정답에 맞춰 예측을 수정하지 않습니다.

2026-10-14 사용자의 Colab 사용 허용에 따라 원격 설치 제한만 해제했습니다. 기존 평가 결과·정책 파일은 보존합니다. 정답과 예측이 포함된 비공개 평가용 노트북이므로 GitHub 공개 업로드 대상이 아닙니다.


In [ ]:
# Install only in Google's remote Colab runtime; no local packages/models.
import sys, subprocess
if 'google.colab' not in sys.modules:
    import importlib.util
    if importlib.util.find_spec('google.colab') is None:
        raise RuntimeError('Run this notebook in a hosted Google Colab runtime.')
subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', 'sentence-transformers'])


In [ ]:
import base64, csv, hashlib, io, json
from pathlib import Path
PAYLOAD = {'predictions.jsonl': {'base64': '...', 'sha256': 'b21dd20e36c6086169417f8fca127d394bf0bd0226f1764605710702e123e1e3'}, 'ground_truth.csv': {'base64': '...', 'sha256': '856f5bc09ac9559cf999c89d9d9962615be704b3a85d2b785317d665f1bb3a91'}}
raw = {}
for name, item in PAYLOAD.items():
    raw[name] = base64.b64decode(item['base64'])
    assert hashlib.sha256(raw[name]).hexdigest() == item['sha256'], name
preds = [json.loads(line) for line in raw['predictions.jsonl'].decode('utf-8-sig').splitlines() if line.strip()]
gold = list(csv.DictReader(io.StringIO(raw['ground_truth.csv'].decode('utf-8-sig'))))
assert len(preds) == 172 and len(gold) == 86
assert len({p['identifier'] for p in preds}) == len(preds)
assert len({g['ruleId'] for g in gold}) == len(gold)
print('Frozen input hashes verified:', len(preds), 'predictions,', len(gold), 'gold rows')


In [ ]:
def hungarian_max(scores):
    """Maximum-weight full assignment, with zero dummy rows/columns.

    Returns real row/column pairs. Threshold filtering is applied afterwards.
    """
    nr = len(scores)
    nc = len(scores[0]) if nr else 0
    if not nr or not nc:
        return []
    if any(len(row) != nc for row in scores):
        raise ValueError("Ragged score matrix")
    n = max(nr, nc)
    cost = [[-scores[i][j] if i < nr and j < nc else 0.0 for j in range(n)] for i in range(n)]
    u, v, p, way = [0.0] * (n + 1), [0.0] * (n + 1), [0] * (n + 1), [0] * (n + 1)
    for i in range(1, n + 1):
        p[0], j0 = i, 0
        minv, used = [float("inf")] * (n + 1), [False] * (n + 1)
        while True:
            used[j0] = True
            i0, delta, j1 = p[j0], float("inf"), 0
            for j in range(1, n + 1):
                if not used[j]:
                    cur = cost[i0 - 1][j - 1] - u[i0] - v[j]
                    if cur < minv[j]:
                        minv[j], way[j] = cur, j0
                    if minv[j] < delta:
                        delta, j1 = minv[j], j
            for j in range(n + 1):
                if used[j]:
                    u[p[j]] += delta
                    v[j] -= delta
                else:
                    minv[j] -= delta
            j0 = j1
            if p[j0] == 0:
                break
        while True:
            j1 = way[j0]
            p[j0] = p[j1]
            j0 = j1
            if j0 == 0:
                break
    return sorted((p[j] - 1, j - 1) for j in range(1, n + 1) if 0 < p[j] <= nr and j <= nc)

def prf(matches, npred, ngold):
    precision = matches / npred if npred else 0.0
    recall = matches / ngold if ngold else 0.0
    return {"matched": matches, "predictions": npred, "gold": ngold,
            "precision": precision, "recall": recall,
            "f1": 2 * precision * recall / (precision + recall) if precision + recall else 0.0}

import importlib.metadata, platform, datetime
import numpy as np
from sentence_transformers import SentenceTransformer
from huggingface_hub import model_info
MODEL = 'sentence-transformers/all-MiniLM-L6-v2'
THRESHOLD = 0.6
revision = model_info(MODEL).sha
model = SentenceTransformer(MODEL, revision=revision, device='cpu')
texts = [p['conditionText'] + ' ' + p['responseText'] for p in preds] + [g['condition'] + ' ' + g['action'] for g in gold]
lengths = [len(model.tokenizer(t, truncation=False)['input_ids']) for t in texts]
emb = model.encode(texts, normalize_embeddings=True, convert_to_numpy=True, show_progress_bar=True)
similarities = emb[len(preds):] @ emb[:len(preds)].T
docs = sorted({p['unit_id'].split(':')[0] for p in preds})
assert all(g['source'] in docs for g in gold)
assignments, accepted, by_doc = [], [], {}
for doc in docs:
    pi = [i for i,p in enumerate(preds) if p['unit_id'].split(':')[0] == doc]
    gi = [i for i,g in enumerate(gold) if g['source'] == doc]
    scores = [[float(similarities[i,j]) for j in pi] for i in gi]
    pairs = hungarian_max(scores)
    count = 0
    for i,j in pairs:
        g,p = gold[gi[i]],preds[pi[j]]
        row = {'document_id':doc, 'gold_id':g['ruleId'], 'prediction_id':p['identifier'],
               'score':scores[i][j], 'accepted':scores[i][j]>=THRESHOLD,
               'gold_class':g['class'], 'prediction_class':p['ruleClass']}
        assignments.append(row)
        if row['accepted']:
            accepted.append(row)
            count += 1
    by_doc[doc] = prf(count,len(pi),len(gi))
mp = {r['prediction_id'] for r in accepted}
mg = {r['gold_id'] for r in accepted}
metrics = {'status':'SBERT_EVALUATION_COMPLETE', 'F1_content':prf(len(accepted),len(preds),len(gold)),
 'method':'condition+action; per-document maximum-weight Hungarian 1:1; threshold after assignment',
 'threshold':THRESHOLD, 'model':MODEL, 'model_revision':revision, 'device':'cpu',
 'hashes':{k:v['sha256'] for k,v in PAYLOAD.items()}, 'by_document':by_doc,
 'unmatched_predictions':[p['identifier'] for p in preds if p['identifier'] not in mp],
 'unmatched_gold':[g['ruleId'] for g in gold if g['ruleId'] not in mg],
 'max_seq_length':model.max_seq_length,
 'truncation':{'predictions':sum(n>model.max_seq_length for n in lengths[:len(preds)]),
               'gold':sum(n>model.max_seq_length for n in lengths[len(preds):]), 'maximum_tokens':max(lengths)},
 'python':platform.python_version(),
 'packages':{k:importlib.metadata.version(k) for k in ['sentence-transformers','transformers','torch','numpy','huggingface-hub']},
 'executed_at_utc':datetime.datetime.now(datetime.timezone.utc).isoformat(),
 'predictions_modified':False, 'openai_api_used':False, 'human_review_completed':False,
 'limitations':['One-to-one metric penalizes split rules; maximum F1 is 2*86/(172+86).',
 'Datasheet predictions have no gold rows and remain in the denominator.',
 'English embedding model with mixed Korean/English inputs; no translation or threshold tuning.',
 'Similarity does not prove numeric, negation, safety, or semantic correctness.']}
result_dir = Path('/content/imaks_sbert_results')
result_dir.mkdir(exist_ok=True)
for name,value in [('metrics.json',metrics),('assignments.json',assignments)]:
    (result_dir/name).write_text(json.dumps(value,ensure_ascii=False,indent=2,allow_nan=False),encoding='utf-8')
np.save(result_dir/'cosine_matrix.npy', similarities)
print(json.dumps(metrics,ensure_ascii=False,indent=2))
print('EVALUATION_COMPLETE')


In [ ]:
import shutil
from google.colab import files
archive = shutil.make_archive('/content/imaks_sbert_results', 'zip', result_dir)
files.download(archive)
